In [1]:

from snakemake.utils import min_version
from pathlib import Path
import pandas as pd
import copy
import xscen as xs
import numpy as np

/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/MPI/gcc12/openmpi4/esmf/8.8.0/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [7]:
configpath = list(Path("../config").glob("*_ESPO.yml"))

In [8]:
import yaml

In [9]:
out = [yaml.safe_load(open(p, 'r')) for p in configpath]

In [10]:
out

[{'subregions': {'n': 50000},
  'full_region': {'name': 'IDM',
   'method': 'bbox',
   'tile_buffer': 1.5,
   'lon_bnds': [-62.5, -61],
   'lat_bnds': [46.9, 48]},
  'extraction': {'reference': {'CaSR': {'search_data_catalogs': {'variables_and_freqs': {'tasmax': 'D',
       'tasmin': 'D',
       'pr': 'D',
       'dtr': 'D'},
      'allow_resampling': False,
      'allow_conversion': True,
      'periods': ['1989', '2023'],
      'other_search_criteria': {'source': 'CaSR', 'version': 'v32'}},
     'extract_dataset': {'mask': False}}},
   'simulation': {'search_data_catalogs': {'variables_and_freqs': {'tasmax': 'D',
      'tasmin': 'D',
      'pr': 'D',
      'dtr': 'D'},
     'match_hist_and_fut': True,
     'allow_conversion': True,
     'allow_resampling': False,
     'periods': ['1951', '2100'],
     'restrict_members': {'ordered': 5},
     'exclusions': {'id': ['CMIP6_ScenarioMIP_CCCma_CanESM5_ssp245_r1i1p1f1_global',
       'CMIP6_ScenarioMIP_CCCma_CanESM5_ssp370_r1i1p1f1_global',

In [11]:
config = xs.config.recursive_update(out[0], out[1])

In [13]:

from copy import deepcopy
args = deepcopy(config["extraction"]["reference"]["CaSR"]["search_data_catalogs"])


In [14]:
args

{'variables_and_freqs': {'tasmax': 'D', 'tasmin': 'D', 'pr': 'D', 'dtr': 'D'},
 'allow_resampling': False,
 'allow_conversion': True,
 'periods': ['1989', '2023'],
 'other_search_criteria': {'source': 'CaSR', 'version': 'v32'},
 'data_catalogs': ['/home/sgammon/espog/pc.json']}

In [1]:
import xarray as xr

In [6]:
xr.open_dataset("/project/ctb-frigon/sgammon/ESPOv2/reference/split_regions/IDM_CaSR_sr-0_default.zarr.zip", engine="zarr", chunks={})

<xarray.Dataset> Size: 14MB
Dimensions:       (time: 12783, loc: 45)
Coordinates:
  * time          (time) datetime64[ns] 102kB 1989-01-01 ... 2023-12-31
    lat           (loc) float32 180B dask.array<chunksize=(45,), meta=np.ndarray>
    lon           (loc) float32 180B dask.array<chunksize=(45,), meta=np.ndarray>
    rlat          (loc) float32 180B dask.array<chunksize=(45,), meta=np.ndarray>
    rlon          (loc) float32 180B dask.array<chunksize=(45,), meta=np.ndarray>
    rotated_pole  int32 4B ...
Dimensions without coordinates: loc
Data variables:
    pr            (time, loc) float64 5MB dask.array<chunksize=(12783, 45), meta=np.ndarray>
    tasmax        (time, loc) float64 5MB dask.array<chunksize=(12783, 45), meta=np.ndarray>
    tasmin        (time, loc) float64 5MB dask.array<chunksize=(12783, 45), meta=np.ndarray>
Attributes: (12/18)
    cat:_data_format_:       nc
    cat:calendar:            default
    cat:date_end:            2024-06-30 00:00:00
    cat:date_start:          1840-03-01 00:00:00
    cat:domain:              IDM
    cat:frequency:           day
    ...                      ...
    cat:variable:            tasmin,tasmax,pr,dtr
    cat:version:             v32
    cat:xrfreq:              D
    history:                 [2026-10-08 14:48:11] bbox spatial subsetting wi...
    intake_esm_dataset_key:  CanHomP-AHCCD_IDM.NAM.raw.D
    intake_esm_vars:         ['pr', 'tasmin', 'tasmax']

In [25]:
cat = xs.search_data_catalogs(**args)

/lustre06/project/6046550/scenario/environnements/envs/xscen-0.15/lib/python3.12/site-packages/xclim/core/indicator.py:296: UserWarning: Class conversions.WIND_SPEED_FROM_VECTOR already exists and will be overwritten.
/lustre06/project/6046550/scenario/environnements/envs/xscen-0.15/lib/python3.12/site-packages/xclim/core/indicator.py:296: UserWarning: Class conversions.WIND_VECTOR_FROM_SPEED already exists and will be overwritten.
/lustre06/project/6046550/scenario/environnements/envs/xscen-0.15/lib/python3.12/site-packages/xclim/core/indicator.py:296: UserWarning: Class conversions.TAS_MIDPOINT already exists and will be overwritten.
/lustre06/project/6046550/scenario/environnements/envs/xscen-0.15/lib/python3.12/site-packages/xclim/core/indicator.py:296: UserWarning: Class conversions.RELATIVE_HUMIDITY_FROM_DEWPOINT already exists and will be overwritten.
/lustre06/project/6046550/scenario/environnements/envs/xscen-0.15/lib/python3.12/site-packages/xclim/core/indicator.py:296: UserW

In [ ]:
dc = cat.popitem()[1]
ds_dict = xs.extract_dataset(
    catalog=dc,
    region=config['full_region'],
    **config["extraction"]["reference"]["CaSR"]["extract_dataset"],
)

In [27]:
ds_dict['D']

<xarray.Dataset> Size: 19MB
Dimensions:       (rlat: 9, rlon: 5, time: 12783)
Coordinates:
  * rlat          (rlat) float32 36B -6.48 -6.39 -6.3 ... -5.94 -5.85 -5.76
  * rlon          (rlon) float32 20B 20.13 20.22 20.31 20.4 20.49
    lat           (rlat, rlon) float32 180B 47.21 47.17 47.13 ... 47.74 47.7
    lon           (rlat, rlon) float32 180B -62.18 -62.06 ... -61.35 -61.24
  * time          (time) datetime64[ns] 102kB 1989-01-01 ... 2023-12-31
    rotated_pole  int32 4B ...
Data variables:
    pr            (rlat, rlon, time) float64 5MB dask.array<chunksize=(9, 5, 12783), meta=np.ndarray>
    tasmax        (rlat, rlon, time) float64 5MB dask.array<chunksize=(9, 5, 12783), meta=np.ndarray>
    tasmin        (rlat, rlon, time) float64 5MB dask.array<chunksize=(9, 5, 12783), meta=np.ndarray>
    dtr           (rlat, rlon, time) float64 5MB dask.array<chunksize=(9, 5, 12783), meta=np.ndarray>
Attributes: (12/17)
    intake_esm_vars:         ['pr', 'tasmax', 'tasmin']
    cat:id:                  CanHomP-AHCCD_IDM
    cat:type:                reconstruction
    cat:processing_level:    extracted
    cat:institution:         GovCan
    cat:source:              CaSR
    ...                      ...
    cat:date_end:            2024-06-30 00:00:00
    cat:version:             v32
    cat:_data_format_:       nc
    cat:path:                /project/ctb-frigon/sgammon/pc-idm-espog2/pc-idm...
    intake_esm_dataset_key:  CanHomP-AHCCD_IDM.NAM.raw.D
    history:                 [2026-10-09 12:12:55] bbox spatial subsetting wi...

In [ ]:
"""Search catalog, extract simulation on the right region and clean it up."""

from copy import deepcopy

import numpy as np
import xarray as xr
import xscen as xs

from workflow.scripts.utils import dask_cluster


if 1 == 0:  # trick vscode
    import snakemake

xr.set_options(
    netcdf_engine_order=[
        "netcdf4",
        "h5netcdf",
    ]
)


    # Get Snakemake parameters
    config = deepcopy(snakemake.config)
    pool = snakemake.wildcards.pool
    output = snakemake.output

    client = dask_cluster(snakemake.params, config["dask"]["client"])

    args = deepcopy(config["extraction"]["simulation"]["search_data_catalogs"])

    # get right sims from pool name
    poolsplit = pool.split("_")
    if "other_search_criteria" not in args:
        args["other_search_criteria"] = {}
    
    if poolsplit[0] == "ScenarioMIP":  # GCM
        args["other_search_criteria"].update({
            "source": poolsplit[1],
            "experiment": poolsplit[2],
        })
    else:  # RCM
        args["other_search_criteria"].update({
            "source": poolsplit[0],
            "experiment": poolsplit[2],
            "driving_model": poolsplit[1],
        })
    # search cat
    cat_sim_id = xs.search_data_catalogs(
        **args,
    )

    # extract all the member
    # FIXME: in a future version, capture tracking_ids here
    # https://github.com/Ouranosinc/ESPO-G/issues/15
    real = []
    for subcat in cat_sim_id.values():
        dict_sim = xs.extract_dataset(
            catalog=subcat,
            region=config["full_region"],
            **config["extraction"]["simulation"]["extract_dataset"],
        )
        ds_r = dict_sim["D"]
        ds_r = ds_r.expand_dims(realization=[ds_r.attrs["cat:id"]])
        real.append(ds_r)

    ds_sim = xr.concat(real, dim="realization")

    # nan weird individual timestep
    if "BCC-CSM2-MR" in pool:  # dtr around -30
        ds_sim["tasmin"] = ds_sim["tasmin"].where(
            ds_sim.time != (ds_sim.time.sel(time="2014-12-31").values),
        )
        ds_sim["dtr"] = ds_sim["dtr"].where(
            ds_sim.time != (ds_sim.time.sel(time="2014-12-31").values),
        )
    if "UKESM1-0-LL_ssp370" in pool:  # 1024mm on QC, replace with spatial neighbor mean
        latcorr = 49.375
        loncorr = 285.9375
        if (ds_sim['lat'].min() <= latcorr
            and ds_sim['lat'].max() >= latcorr
            and ds_sim['lon'].min() <= loncorr
            and ds_sim['lon'].max() >= loncorr
           ):
                
            ilat = ds_sim["lat"].values.tolist().index(49.375)
            ilon = ds_sim["lon"].values.tolist().index(285.9375)
            mean_around = (
                ds_sim.sel(
                    time="2100-08-11",
                    realization="CMIP6_ScenarioMIP_MOHC_UKESM1-0-LL_ssp370_r1i1p1f2_global",
                )
                .isel(
                    lat=slice(ilat - 1, ilat + 2),
                    lon=slice(ilon - 1, ilon + 2),
                )
                .squeeze()
                .pr.values
            )
            mean_around[1, 1] = np.nan
            fillval = np.nanmean(mean_around)
            ds_sim["pr"] = ds_sim["pr"].where(
                ~(
                    (ds_sim.lat == 49.375)
                    & (ds_sim.lon == 285.9375)
                    & (ds_sim.time == ds_sim.time.sel(time="2100-08-11").values)
                    & (
                        ds_sim.realization
                        == "CMIP6_ScenarioMIP_MOHC_UKESM1-0-LL_ssp370_r1i1p1f2_global"
                    )
                ),
                other=fillval,
            )

    # https://errata.esgf.io/static/view.html?uid=76b3f818-d65f-c76b-bfd8-cae5bc27825c
    if "UKESM1-0-LL" in pool:
        ds_sim["tasmax"] = ds_sim["tasmax"].where(
            ds_sim.tasmax <= 335,
        )

    # clean up time
    ds_sim["time"] = ds_sim.time.dt.floor("D")

    if "mask" not in ds_sim and "create_mask" in config["extraction"]["simulation"]:
        ds_sim["mask"] = xs.regrid.create_mask(
            dict_sim["fx"], **config["extraction"]["simulation"]["create_mask"]
        )

    ds_sim = xs.clean_up(ds_sim, **config["extraction"]["clean_up"])

    # save to zarr
    xs.save_to_zarr(
        ds_sim,
        output["extract"],
        rechunk=config["chunks"]["pre-regrid"],
        **config["save_to_zarr"],
    )
